# Ridge Regression

## What is Ridge Regression?

Ridge Regression is a **regularized version of Linear Regression**.

It is useful when Linear Regression becomes too sensitive to the training data or when some coefficients become very large.

The main idea is simple:

> **Make good predictions while keeping the coefficients under control.**

---

## Why Do We Need Ridge Regression?

Linear Regression tries to minimize prediction error.

Sometimes this can result in:

* Large coefficients
* Sensitivity to small changes in data
* Problems when features are highly correlated
* Overfitting
* Poor performance on unseen data

Ridge Regression adds a **penalty for large coefficients**.

This penalty is called **L2 Regularization**.

---

## Ridge Regression Intuition

### Linear Regression

**Goal:** Minimize prediction error.

### Ridge Regression

**Goal:** Minimize prediction error + penalize large coefficients.

[
\text{Ridge Loss} = \text{MSE} + \alpha \sum_{j=1}^{p}\beta_j^2
]

Where:

* **MSE** = prediction error
* **β (beta)** = model coefficients
* **α (alpha)** = regularization strength
* **Σβ²** = penalty applied to large coefficients

The intercept is normally not included in the regularization penalty.

---

## Understanding Alpha

`alpha` controls how strongly Ridge penalizes large coefficients.

| Alpha               | Effect                             |
| ------------------- | ---------------------------------- |
| `α ≈ 0`             | Behaves close to Linear Regression |
| Small `α`           | Weak regularization                |
| Medium `α`          | More coefficient shrinkage         |
| Large `α`           | Strong regularization              |
| Extremely large `α` | Can cause underfitting             |

### Important

Increasing Polynomial degree and increasing Ridge alpha have almost opposite effects:

**Polynomial degree ↑ → model complexity ↑**

**Ridge alpha ↑ → model flexibility ↓**

---

## What Does Ridge Do to Coefficients?

Suppose Linear Regression produces:

```text
x1 = 2
x2 = -5
x3 = 800
x4 = -950
```

Ridge may shrink them toward smaller values:

```text
x1 = 1.8
x2 = -4.2
x3 = 120
x4 = -160
```

The exact values depend on the dataset, feature scaling, and alpha.

### Important

Ridge generally **shrinks coefficients toward zero**.

It does **not normally make coefficients exactly zero**.

This is an important difference between **Ridge and Lasso Regression**.

---

# Why Scaling Matters

Ridge penalizes model coefficients.

If features are measured on very different scales, the coefficient sizes are not directly comparable.

Example:

```text
Age        = 25
Salary     = 50000
House Size = 1800
Rooms      = 3
```

Therefore, Ridge is commonly used with feature scaling such as:

```python
StandardScaler()
```

Typical workflow:

```text
Dataset
   ↓
Train/Test Split
   ↓
StandardScaler
   ↓
Ridge Regression
   ↓
Prediction
   ↓
Evaluation
```

Using a `Pipeline` makes this workflow easier and helps ensure preprocessing is applied consistently.

---

# Ridge vs Linear Regression

Suppose:

```text
Linear Regression

Train R² = 0.97
Test R²  = 0.71
```

This may indicate overfitting.

After Ridge:

```text
Ridge Regression

Train R² = 0.91
Test R²  = 0.86
```

Even though training performance decreased, test performance improved.

This is often desirable because our goal is not to achieve the highest training score.

Our goal is to **generalize well to unseen data**.

---

# When Should I Consider Ridge Regression?

Consider Ridge when:

* Linear Regression is overfitting
* Features are highly correlated
* Coefficients are unstable or very large
* There are many input features
* Polynomial feature expansion creates many correlated features
* You want to retain all features while reducing their influence

---

# Ridge Regression Workflow

```text
1. Load Dataset
        ↓
2. Understand / Clean Data
        ↓
3. Separate X and y
        ↓
4. Train/Test Split
        ↓
5. Scale Features
        ↓
6. Train Ridge
        ↓
7. Predict
        ↓
8. Evaluate MAE / RMSE / R²
        ↓
9. Try different alpha values
        ↓
10. Compare with Linear Regression
```

---

## Today's Practical Goal

We will:

1. Load a real regression dataset
2. Create a Linear Regression baseline
3. Build Ridge Regression
4. Use `StandardScaler` with a `Pipeline`
5. Evaluate train and test performance
6. Experiment with different `alpha` values
7. Observe coefficient shrinkage
8. Compare Linear Regression vs Ridge Regression
9. Understand underfitting and overfitting
10. Decide which model generalizes better

---

## Key Idea to Remember

> **Linear Regression:** Find coefficients that minimize prediction error.

> **Ridge Regression:** Find coefficients that minimize prediction error while also keeping the coefficients small.

### Memory Shortcut

```text
Large / unstable coefficients
          ↓
        Ridge
          ↓
L2 Regularization
          ↓
Shrink coefficients
          ↓
Control model flexibility
          ↓
Potentially better generalization
```


# Practical - Diabetes Dataset

In [9]:
from ucimlrepo import fetch_ucirepo

concrete = fetch_ucirepo(id=165)

X = concrete.data.features
y = concrete.data.targets.squeeze()

In [10]:
X.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360


In [11]:
y.head()

0    79.99
1    61.89
2    40.27
3    41.05
4    44.30
Name: Concrete compressive strength, dtype: float64

In [12]:
X.isnull().sum()

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
dtype: int64

In [13]:
X.isnull().any()

Cement                False
Blast Furnace Slag    False
Fly Ash               False
Water                 False
Superplasticizer      False
Coarse Aggregate      False
Fine Aggregate        False
Age                   False
dtype: bool

In [14]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1030, 8)
y shape: (1030,)


In [15]:
X.describe()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000


In [18]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size= 0.2,
    random_state= 42
)

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
lin_model = Pipeline([
    ("scalar", StandardScaler()),
    ("model", LinearRegression())
])

In [25]:
lin_model.fit(X_train,y_train)

Pipeline(steps=[('scalar', StandardScaler()), ('model', LinearRegression())])

In [26]:
y_pred = lin_model.predict(X_train)
y_test_pred = lin_model.predict(X_test)

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def regression_score(model, X, y):
    
    pred = model.predict(X)
    
    print("R²   :", round(r2_score(y, pred), 3))
    print("MAE  :", round(mean_absolute_error(y, pred), 3))
    print("RMSE :", round(np.sqrt(mean_squared_error(y, pred)), 3))

In [28]:
regression_score(lin_model, X_train, y_train)

R²   : 0.611
MAE  : 8.33
RMSE : 10.519


In [29]:
regression_score(lin_model, X_test, y_test)

R²   : 0.628
MAE  : 7.746
RMSE : 9.796


In [30]:
from sklearn.linear_model import Ridge

ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

In [31]:
ridge_model.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', Ridge())])

In [32]:
y_pred = ridge_model.predict(X_train)
y_test_pred = ridge_model.predict(X_test)

In [33]:
regression_score(ridge_model, X_train, y_train)

R²   : 0.61
MAE  : 8.337
RMSE : 10.519


In [34]:
regression_score(ridge_model, X_test, y_test)

R²   : 0.628
MAE  : 7.752
RMSE : 9.796


In [35]:
print("RIDGE TRAIN")
regression_score(ridge_model, X_train, y_train)

print("\nRIDGE TEST")
regression_score(ridge_model, X_test, y_test)

RIDGE TRAIN
R²   : 0.61
MAE  : 8.337
RMSE : 10.519

RIDGE TEST
R²   : 0.628
MAE  : 7.752
RMSE : 9.796


In [36]:
for alpha in [0.01, 0.1, 1, 10, 100, 1000]:

    ridge_model = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])

    ridge_model.fit(X_train, y_train)

    train_score = ridge_model.score(X_train, y_train)
    test_score = ridge_model.score(X_test, y_test)

    print(
        "Alpha:", alpha,
        "| Train R²:", round(train_score, 3),
        "| Test R²:", round(test_score, 3)
    )

Alpha: 0.01 | Train R²: 0.611 | Test R²: 0.628
Alpha: 0.1 | Train R²: 0.611 | Test R²: 0.628
Alpha: 1 | Train R²: 0.61 | Test R²: 0.628
Alpha: 10 | Train R²: 0.609 | Test R²: 0.626
Alpha: 100 | Train R²: 0.589 | Test R²: 0.605
Alpha: 1000 | Train R²: 0.42 | Test R²: 0.425


In [37]:
linear_coef = lin_model.named_steps["model"].coef_

print(linear_coef)

[12.78841262  9.43445595  5.25457769 -2.88259683  1.85212598  1.40519554
  1.9505291   7.03743401]


In [38]:
ridge_100 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=100))
])

ridge_100.fit(X_train, y_train)

ridge_coef = ridge_100.named_steps["model"].coef_

print(ridge_coef)

[ 7.25736831  4.16005358  0.60478636 -4.71739124  2.64374147 -1.36367351
 -1.83167313  5.91159029]


In [39]:
import pandas as pd

coef_compare = pd.DataFrame({
    "Feature": X.columns,
    "Linear": linear_coef,
    "Ridge": ridge_coef
})

coef_compare

,Feature,Linear,Ridge
0,Cement,12.788413,7.257368
1,Blast Furnace Slag,9.434456,4.160054
2,Fly Ash,5.254578,0.604786
3,Water,-2.882597,-4.717391
4,Superplasticizer,1.852126,2.643741
5,Coarse Aggregate,1.405196,-1.363674
6,Fine Aggregate,1.950529,-1.831673
7,Age,7.037434,5.911590


In [40]:
import numpy as np

print(
    "Linear coefficient magnitude:",
    np.linalg.norm(linear_coef)
)

print(
    "Ridge coefficient magnitude:",
    np.linalg.norm(ridge_coef)
)

Linear coefficient magnitude: 18.633501922018404
Ridge coefficient magnitude: 11.821416493838415
